In [111]:
import os, pdfplumber, re, json, hashlib, unicodedata
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [112]:
# -- Configurations --
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
OUTPUT_FILE = "mental_health_cleaned.jsonl"

DOCS_CONFIG = [
    {"path": "resources/pdtt.pdf", "type": "medical_guideline", "risk": "high"},
    {"path": "resources/sctl.pdf", "type": "first_aid", "risk": "medium"},
    {"path": "resources/tvtl.pdf", "type": "school_counseling", "risk": "medium"}
]

In [113]:
# -- Helper Functions --
def clean_text(text):
    if not text: return ""
    # Normalize Unicode
    text = unicodedata.normalize("NFC", text)
    # Fix common OCR errors specific to the documents
    fix_map = {
        "SÖT": "SÚT", "HƢỚNG": "HƯỚNG", "Ƣ": "Ư", "thƣ": "thư", 
        "TRỊ": "TRỊ", "đối tƣợng": "đối tượng", "tƣ": "tư", "Vi\u0000t": "Việt", "chi\u0000n": "chiến"
    }
    for k, v in fix_map.items(): 
        text = text.replace(k, v)
    # Delete page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    # Fix hyphenated words at line breaks
    text = re.sub(r'(\w+)-\n\s*(\w+)', r'\1\2', text)
    # Replace multiple spaces/tabs with a single space
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def extract_tables_to_md(page):
    # Extract tables and convert to Markdown format
    tables = page.extract_tables()
    table_md = ""
    if tables:
        for table in tables:
            table_md += "\n\n[DỮ LIỆU BẢNG TRA CỨU]:\n"
            for row in table:
                if any(row):
                    # Clean each cell and replace empty cells with "-"
                    r = [clean_text(str(c)) if c else "-" for c in row]
                    table_md += "| " + " | ".join(r) + " |\n"
    return table_md

def get_section_title(text, current_title):
    # Try to find section titles based on common patterns in the documents
    section_patterns = [
        r'(Bài\s+\d+[:\.]?.*)', 
        r'(Chương\s+\d+[:\.]?.*)', 
        r'(Phần\s+[IVX]+[:\.]?.*)',
        r'(Phụ lục\s+\d+[:\.]?.*)',
        r'(\d+\.\d+\.?\s+[A-ZĐ].*)'
    ]
    for pat in section_patterns:
        found = re.search(pat, text)
        if found:
            new_title = found.group(1).strip()
            # Only update title if it's reasonably short
            if len(new_title) < 100:
                return new_title
    return current_title

In [114]:
# -- Main Chunking Logic --
def run_chunking():
    final_data = []
    seen_hashes = set()
    # Initialize the text splitter 
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n[MỤC:", "\n\n", "\n", ". ", " ", ""]
    )
    for doc in DOCS_CONFIG:
        if not os.path.exists(doc['path']):
            print(f"Không tìm thấy file: {doc['path']}")
            continue
        print(f"Processing: {doc['path']}")
        with pdfplumber.open(doc['path']) as pdf:
            current_section = "Thông tin chung"
            for page in pdf.pages:
                raw_text = page.extract_text() or ""
                # Skip table of contents or index pages based on common patterns
                if "MỤC LỤC" in raw_text.upper() or len(re.findall(r'\.{10,}', raw_text)) > 5:
                    continue
                cleaned_text = clean_text(raw_text)
                current_section = get_section_title(cleaned_text, current_section)
                # Extract tables and append to content
                table_content = extract_tables_to_md(page)
                # Combine text and table content, prefixed with source and section info
                full_content = f"[NGUỒN: {doc['path']}] - [MỤC: {current_section}]\n{cleaned_text}\n{table_content}"
                # Split into chunks
                chunks = splitter.split_text(full_content)
                for chunk in chunks:
                    # Remove chunks that are too short to be meaningful
                    if len(chunk.strip()) < 120:
                        continue
                    # Use hash to avoid duplicates
                    text_hash = hashlib.md5(chunk.encode()).hexdigest()
                    if text_hash in seen_hashes: continue
                    seen_hashes.add(text_hash)
                    # Append to final data with metadata
                    final_data.append({
                        "page_content": chunk.strip(),
                        "metadata": {
                            "source": doc['path'],
                            "section": current_section,
                            "risk_priority": doc['risk'],
                            "doc_type": doc['type'],
                            "page_no": page.page_number
                        }
                    })
    # Save to JSONL file
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for entry in final_data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            
    print(f"Saved {len(final_data)} chunks to {OUTPUT_FILE}")

# Execute the chunking process
if __name__ == "__main__":
    run_chunking()

Processing: resources/pdtt.pdf
Processing: resources/sctl.pdf
Processing: resources/tvtl.pdf
Saved 1042 chunks to mental_health_cleaned.jsonl
